# Raw to Bronze — VRA/ANAC

Cria a camada **Bronze** do VRA como *external table* sobre o GCS.
Não há leitura para pandas nem tipagem: a Bronze mantém
os dados exatamente como chegam da fonte, com todos os campos `STRING`.
A tipagem explícita fica para o notebook/SQL da Silver.

Passos:
1. Reorganiza os CSVs já enviados ao bucket para o layout particionado por
   data de ingestão / lote.
2. Cria a external table com o schema do dicionário de dados.
3. Valida a contagem de linhas contra o manifest gerado na ingestão.

In [ ]:
from datetime import date
from pathlib import Path
import json

from google.cloud import bigquery
from google.cloud import storage

## Configuração dos parâmetros

In [ ]:
PROJECT_ID = "pdm-bia-2026"
DATASET_ID = "tf_anac"
TABLE_NAME = "tb_anac_bronze"

BUCKET_NAME = "dados-anac-vra"
RAW_PREFIX = "bronze/vra/raw"

# Data em que os CSVs foram enviados ao bucket e identificador do lote —
# confirme contra o manifest em manifests/ se for rodar para um novo recorte.
INGESTED_AT = date(2026, 9, 3)
BATCH_ID = "vra_2022_2024"

FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"
GCS_URI_PREFIX = f"gs://{BUCKET_NAME}/{RAW_PREFIX}"

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

print(f"Tabela de destino: {FULL_TABLE_ID}")
print(f"Origem dos dados: {GCS_URI_PREFIX}/*.csv")

Tabela de destino: pdm-bia-2026.tf_anac.tb_anac_bronze
Origem dos dados: gs://dados-anac-vra/bronze/vra/raw/*.csv


## Reorganiza os arquivos no GCS (layout Hive por data de ingestão / lote)

Move os CSVs de `raw/VRA_*.csv` (layout plano) para
`raw/_ingested_at=<data>/_batch_id=<lote>/VRA_*.csv`. Idempotente: só move
o que ainda está no caminho plano, então pode ser executada de novo sem
duplicar nada.

In [ ]:
bucket = storage_client.bucket(BUCKET_NAME)
destino_prefixo = f"{RAW_PREFIX}/_ingested_at={INGESTED_AT.isoformat()}/_batch_id={BATCH_ID}/"

movidos = 0
for blob in list(bucket.list_blobs(prefix=f"{RAW_PREFIX}/")):
    nome = blob.name.split("/")[-1]
    ja_particionado = "_ingested_at=" in blob.name
    if not nome.endswith(".csv") or ja_particionado:
        continue
    bucket.rename_blob(blob, f"{destino_prefixo}{nome}")
    movidos += 1

print(f"Arquivos movidos para o layout particionado: {movidos}")

Arquivos movidos para o layout particionado: 0


## Definição do schema e criação da tabela Bronze (external table)

Ordem e nomes de coluna seguem o dicionário de dados verificado do — o mapeamento é posicional, já que `skip_leading_rows` não casa
por nome de cabeçalho. Todos os campos `STRING`.

In [ ]:
schema = [
    bigquery.SchemaField("sg_empresa_icao", "STRING", description="Código ICAO da companhia operadora"),
    bigquery.SchemaField("nm_empresa", "STRING", description="Razão social da companhia"),
    bigquery.SchemaField("nr_voo", "STRING", description="Número do voo atribuído pela companhia"),
    bigquery.SchemaField("cd_di", "STRING", description="Dígito identificador do tipo de operação"),
    bigquery.SchemaField("cd_tipo_linha", "STRING", description="Natureza da linha: N nacional, I internacional, C cargueiro, G -"),
    bigquery.SchemaField("sg_equipamento_icao", "STRING", description="Modelo da aeronave (ex.: B77W, A20N)"),
    bigquery.SchemaField("nr_assentos_ofertados", "STRING", description="Assentos ofertados"),
    bigquery.SchemaField("sg_icao_origem", "STRING", description="Aeroporto de partida"),
    bigquery.SchemaField("nm_aerodromo_origem", "STRING", description="Nome do aeroporto de partida"),
    bigquery.SchemaField("dt_partida_prevista", "STRING", description="Horário programado de partida"),
    bigquery.SchemaField("dt_partida_real", "STRING", description="Horário efetivo de partida — proibido como feature"),
    bigquery.SchemaField("sg_icao_destino", "STRING", description="Aeroporto de chegada"),
    bigquery.SchemaField("nm_aerodromo_destino", "STRING", description="Nome do aeroporto de chegada"),
    bigquery.SchemaField("dt_chegada_prevista", "STRING", description="Horário programado de chegada"),
    bigquery.SchemaField("dt_chegada_real", "STRING", description="Horário efetivo de chegada — proibido"),
    bigquery.SchemaField("ds_situacao_voo", "STRING", description="REALIZADO ou CANCELADO — proibido"),
    bigquery.SchemaField("ds_justificativa", "STRING", description="Motivo do atraso — vem vazio na base observada — proibido"),
    bigquery.SchemaField("dt_referencia", "STRING", description="Data de referência do voo"),
    bigquery.SchemaField("ds_situacao_partida", "STRING", description="Faixa oficial ANAC do atraso de partida — proibido como feature, só para derivar o rótulo"),
    bigquery.SchemaField("ds_situacao_chegada", "STRING", description="Idem, para a chegada — proibido"),
    bigquery.SchemaField("ds_codeshare", "STRING", description="Indicador de voo codeshare — só presente a partir de out/2022; NULL nos arquivos sem a 21ª coluna"),
]

Particionamento Hive (`_ingested_at`, `_batch_id`) satisfaz. Uma
*external table* não aceita `CLUSTER BY` no BigQuery — clusterização fica para a Silver, que já é tabela nativa.

In [ ]:
external_config = bigquery.ExternalConfig("CSV")
external_config.source_uris = [f"{GCS_URI_PREFIX}/*"]
external_config.schema = schema
external_config.options.field_delimiter = ";"
external_config.options.skip_leading_rows = 1
external_config.options.encoding = "ISO-8859-1"
external_config.options.allow_jagged_rows = True

hive_partitioning = bigquery.external_config.HivePartitioningOptions()
hive_partitioning.mode = "AUTO"
hive_partitioning.source_uri_prefix = f"{GCS_URI_PREFIX}/"
hive_partitioning.require_partition_filter = False
external_config.hive_partitioning = hive_partitioning

table = bigquery.Table(FULL_TABLE_ID, schema=schema)
table.external_data_configuration = external_config
table.description = (
    "Camada Bronze do VRA/ANAC. External table sobre o GCS, todos os campos "
    "STRING, append-only. Particionada por _ingested_at/_batch_id "
    "via layout Hive. allow_jagged_rows=True por causa da coluna "
    "Codeshare, ausente antes de out/2022."
)

Cria o dataset (se não existir) e recria a tabela — recriação idempotente,
equivalente em intenção ao `WRITE_TRUNCATE` do exemplo de referência, mas
aqui é só metadado: nenhum dado é copiado, é *schema-on-read*.

In [ ]:
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
bq_client.create_dataset(bigquery.Dataset(dataset_ref), exists_ok=True)

bq_client.delete_table(FULL_TABLE_ID, not_found_ok=True)
table = bq_client.create_table(table)

print(f"Tabela externa criada: {table.full_table_id}")

Tabela externa criada: pdm-bia-2026:tf_anac.tb_anac_bronze


## Validação

Confere a contagem de linhas da Bronze contra o manifest gerado na
ingestão. Também mostra quantas linhas vieram com `ds_codeshare` nulo
(arquivos de 20 colunas) vs. preenchido (arquivos de 21 colunas).

In [ ]:
query = f"""
SELECT
  _batch_id,
  _ingested_at,
  COUNT(*) AS linhas,
  COUNT(DISTINCT _FILE_NAME) AS arquivos,
  COUNTIF(ds_codeshare IS NULL) AS linhas_sem_codeshare
FROM `{FULL_TABLE_ID}`
GROUP BY 1, 2
ORDER BY 1, 2
"""
resultado = bq_client.query(query).to_dataframe()
resultado

,_batch_id,_ingested_at,linhas,arquivos,linhas_sem_codeshare
0,vra_2022_2024,2026-09-03,2843883,36,2333003


In [ ]:
manifest_blob_name = f"bronze/vra/manifest/vra_bronze_manifest_{BATCH_ID.removeprefix('vra_')}.json"
manifest_content = bucket.blob(manifest_blob_name).download_as_text()
manifest = json.loads(manifest_content)

total_manifest = sum(item["linhas_dados"] for item in manifest)
total_bronze = int(resultado["linhas"].sum())

print(f"Linhas no manifest: {total_manifest}")
print(f"Linhas na Bronze:  {total_bronze}")
assert total_manifest == total_bronze, "Divergência entre o manifest de ingestão e a Bronze!"
print("Contagem confere.")

Linhas no manifest: 2843883
Linhas na Bronze:  2843883
Contagem confere.
